# NanoGPT — ROCStories Story Generation
## COMP4680/8650 Advanced Topics in Machine Learning

**Student:** Adithya Rama | **ANU** | March 2026

### What this notebook does
This notebook trains and evaluates a series of nanoGPT models on the ROCStories dataset
for 5-sentence story generation. The pipeline covers:

- **Task 1:** Train the official nanoGPT "baby GPT" baseline (6L/6H/384D, ~30M params)
- **Task 2:** 5-way architecture ablation — isolating RoPE, RMSNorm+SwiGLU, and QK-Norm
- **Task 3:** Best submission model (All Modern + mixed instruction data + TinyStories)
- **Task 4:** Arena model (152M params, combined corpus, unconstrained)

### Key findings (spoiler)
1. All architecture modifications stay ≤32M params (SwiGLU uses 8/3× hidden, not 4×)
2. QK-Norm (+QK-Norm, Config D) is the only modification that individually improves PPL
3. RoPE underperforms at 256-token context — its advantage requires longer sequences
4. Data volume (adding TinyStories) improves PPL more than any architectural change
5. Best checkpoint is at step 1250–2250 for pure ROCStories; overfitting sets in after

### Architecture constraint
All Tasks 1–3 models use ≤32M parameters per assignment specification.
Task 4 (arena competition) is unconstrained.

---
| Section | Description |
|---------|-------------|
| §0 | Environment setup (install, GPU check, repo) |
| §1 | **Task 1** — Official baseline (6L/6H/384D, ~30 M) |
| §2 | **Task 2** — Architecture ablations + instruction-tuning experiment |
| §3 | **Task 3** — Best ≤32 M checkpoint + HuggingFace upload |
| §4 | **Task 4** — Arena model (152 M, optional, no size limit) |
| §5 | **Summary** — Aggregated results, charts, and report table |

## §0 · Environment Setup

In [1]:
# Install required packages (tiktoken for BPE tokenisation, datasets for HuggingFace)
!pip install -q tiktoken datasets huggingface_hub

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    # TF32 gives free ~20% speedup on Ampere GPUs with negligible precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32  = True
print("\u2713 Dependencies ready")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA L4
VRAM    : 23.7 GB
✓ Dependencies ready


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# ── Edit these two variables to match your setup ─────────────────────────────
REPO_URL  = "https://github.com/Adithya-Rama/nano-llm.git"   # your GitHub repo
LOCAL_DIR = "/content/drive/MyDrive/COMP8650/Assgn-1/nano-llm/code-v2"

if not os.path.exists(LOCAL_DIR):
    !git clone "{REPO_URL}" "{LOCAL_DIR}"
else:
    !cd "{LOCAL_DIR}" && git pull

os.chdir(LOCAL_DIR)
print(f"\u2713 Working directory: {os.getcwd()}")

Mounted at /content/drive
Already up to date.
✓ Working directory: /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code-v2


In [3]:
import subprocess, time, sys

def run_streaming(cmd, label=None):
    """
    Run a shell command and stream its output line-by-line to the cell in real-time.
    Uses subprocess.Popen so training progress (iter X | loss Y | lr Z) is visible
    immediately rather than appearing all at once when the process finishes.
    Returns the exit code.
    """
    if label:
        print(f"\n{'━' * 62}")
        print(f"  ▶  {label}")
        print(f"{'━' * 62}\n")
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # merge stderr → same stream as stdout
        text=True,
        bufsize=1,                  # line-buffered for immediate flushing
    )
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    icon = "✓" if proc.returncode == 0 else f"✗ exit {proc.returncode}"
    print(f"\n{'━' * 62}")
    print(f"  {icon}  finished in {mins}m {secs}s")
    print(f"{'━' * 62}")
    return proc.returncode

print("✓  run_streaming() helper ready — all training cells will now stream output in real-time")

✓  run_streaming() helper ready — all training cells will now stream output in real-time


In [ ]:
# ── §0.2 Dataset size verification ────────────────────────────────────────
# Run this BEFORE training to confirm all datasets are built correctly.
# mixed/train.bin should be >10M tokens (ROCStories + TinyStories).
# If it shows <4M, re-run: python data/mixed/prepare.py --with_tinystories
import numpy as np, os, math
os.chdir(LOCAL_DIR)

print("── Dataset sizes ──────────────────────────────────────────────────")
datasets = {
    'rocstories/train':   'data/rocstories/train.bin',
    'rocstories/val':     'data/rocstories/val.bin',
    'tinystories/train':  'data/tinystories/train.bin',
    'mixed/train':        'data/mixed/train.bin',
    'mixed/val':          'data/mixed/val.bin',
}
for name, path in datasets.items():
    if os.path.exists(path):
        n = len(np.fromfile(path, dtype=np.uint16))
        print(f"  {name:<25} {n/1e6:>7.1f}M tokens")
    else:
        print(f"  {name:<25} NOT FOUND")

mixed_path = 'data/mixed/train.bin'
if os.path.exists(mixed_path):
    n = len(np.fromfile(mixed_path, dtype=np.uint16))
    if n < 10_000_000:
        print(f"\n⚠ mixed/train.bin has only {n/1e6:.1f}M tokens — TinyStories NOT included!")
        print("  Run: python data/mixed/prepare.py --with_tinystories")
    else:
        print(f"\n✓ mixed/train.bin looks correct ({n/1e6:.1f}M tokens — TinyStories present)")

In [ ]:
# ── §0.3 Best PPL tracker utility ─────────────────────────────────────────
# Run this cell any time to see a snapshot of all completed training runs.
# Reads train_log.jsonl from each output directory.
import json, math, os
os.chdir(LOCAL_DIR)

def get_best_ppl(out_dir, label=""):
    log_path = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log_path):
        print(f"  {label:<30} log not found")
        return None, None
    entries = [json.loads(l) for l in open(log_path) if 'val_loss' in l]
    if not entries:
        return None, None
    best  = min(entries, key=lambda e: e['val_loss'])
    final = max(entries, key=lambda e: e['step'])
    return best, final

def summarise_run(out_dir, label):
    best, final = get_best_ppl(out_dir, label)
    if best is None:
        print(f"  {label:<30} not trained yet")
        return
    b_ppl = math.exp(best['val_loss'])
    f_ppl = math.exp(final['val_loss'])
    marker = ' ✓' if b_ppl < 25 else '  '
    print(f"  {label:<30} best={b_ppl:.1f}{marker} @step {best['step']:<5} | "
          f"final={f_ppl:.1f} @step {final['step']}")

print("── Best PPL recorded across all runs ──────────────────────────────")
summarise_run('out-t1-baseline',   'T1 Baseline')
summarise_run('out-t2-vanilla',    'T2-A Vanilla')
summarise_run('out-t2-rope',       'T2-B +RoPE')
summarise_run('out-t2-ffn',        'T2-C +RMSNorm+SwiGLU')
summarise_run('out-t2-qknorm',     'T2-D +QK-Norm ★')
summarise_run('out-t2-all-modern', 'T2-E All Modern')
summarise_run('out-t3-best',       'T3 Best Submission')
summarise_run('out-t4-arena',      'T4 Arena (152M)')

---
## §1 · Task 1 — Baseline nanoGPT on ROCStories

### 1.1 Data Pipeline

**Dataset:** ROCStories (Mostafazadeh et al., 2016) — 78,528 five-sentence commonsense stories.
Downloaded from `mintujupally/ROCStories` on HuggingFace.

**Preprocessing steps:**
1. Stories are concatenated as plain text, separated by the GPT-2 `<|endoftext|>` token (id 50256).
2. Tokenised with `tiktoken` GPT-2 BPE (vocab = 50,257; padded to 50,304 for CUDA efficiency).
3. Split: 90% train (~3.7 M tokens) / 10% val (~410 K tokens) — fixed seed 42.
4. Written as raw `uint16` binary files (`train.bin`, `val.bin`) for fast memory-mapped loading during training.

> **No test stories are used during training.** The 19,633-story public test set is only used for final PPL evaluation.

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

print("Preparing ROCStories (plain format for Task 1 + Task 3)...")
!python data/rocstories/prepare.py

# Verify output
for split in ['train', 'val']:
    p = f'data/rocstories/{split}.bin'
    n = len(np.fromfile(p, dtype=np.uint16))
    print(f"  {p}: {n:,} tokens")

Preparing ROCStories (plain format for Task 1 + Task 3)...
[prepare] Trying to load: mintujupally/ROCStories ...
README.md: 100% 256/256 [00:00<00:00, 1.10MB/s]
Repo card metadata block was not found. Setting CardData to empty.
train.txt: 100% 18.1M/18.1M [00:00<00:00, 28.8MB/s]
test.txt: 4.52MB [00:00, 47.7MB/s]
Generating train split: 100% 78528/78528 [00:00<00:00, 250697.25 examples/s]
Generating test split: 100% 19633/19633 [00:00<00:00, 376136.31 examples/s]
[prepare] Columns: ['text']
[prepare] Loaded 78,528 stories from mintujupally/ROCStories (plain format)
[prepare] Split: 70,676 train | 7,852 val

[prepare] Sample story:
The boy went to a video arcade. He played his favorite machine. His games didn't go very well. He told the owner about his experience. The owner explained that he had made the game settings harder.

[prepare] train.bin: 3,701,102 tokens  (avg 52.4 tok/story)  -> /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm

### 1.2 Model Architecture

Following the professor's specification, the **Task 1 baseline** uses the official nanoGPT *baby GPT* configuration:

| Hyperparameter | Value |
|----------------|-------|
| `n_layer` | 6 |
| `n_head` | 6 |
| `n_embd` | 384 |
| `block_size` | 256 |
| `vocab_size` | 50,304 (GPT-2 BPE, padded) |
| **Total params** | **~30.2 M** (≤ 32 M ✓) |

Architecture is vanilla nanoGPT: learned positional embeddings, LayerNorm, GELU MLP — identical to the official `train_shakespeare_char.py` config (Karpathy, 2022).

**Reference:** Karpathy, A. (2022). nanoGPT. https://github.com/karpathy/nanoGPT

In [ ]:
import torch, sys
sys.path.insert(0, LOCAL_DIR)
from model import GPT, GPTConfig

cfg = GPTConfig(
    n_layer=6, n_head=6, n_embd=384, block_size=256,
    vocab_size=50304, bias=False, dropout=0.1,
    use_rmsnorm=False, use_rope=False,
    use_swiglu=False, use_qk_norm=False
)
model = GPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {n_params/1e6:.2f} M")
assert n_params <= 32e6, f"EXCEEDS 32M limit! ({n_params/1e6:.1f}M)"
print(f"Within 32M limit : \u2713")
del model; torch.cuda.empty_cache()

number of parameters: 29.94M
Total parameters : 30.04 M
Within 32M limit : ✓


### 1.3 Training

**Key hyperparameters** (see `config/train_t1_baseline.py`):

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `learning_rate` | 6e-4 | Standard peak LR for ~30 M GPT (Kaplan et al., 2020) |
| `max_iters` | 10,000 | ~88 passes through ROCStories |
| `batch_size` | 32 | Effective batch = 32 × 4 × 256 = 32,768 tokens/step |
| `warmup_iters` | 200 | Short warmup; small dataset |
| `label_smoothing` | 0.1 | Prevents overconfident predictions on small data |
| `dropout` | 0.1 | Regularisation |
| `dtype` | bfloat16 | Native A100 format; no quality loss |

LR schedule: cosine decay from 6e-4 → 6e-5.  
Checkpointing: every 15 minutes (time-based) + every eval_interval (250 steps) to survive Colab disconnects.

**Reference:** Kaplan, J. et al. (2020). Scaling laws for neural language models. arXiv:2001.08361.

In [ ]:
import os
os.chdir(LOCAL_DIR)

T1_CONFIG  = 'config/train_t1_baseline.py'
T1_OUT_DIR = 'out-t1-baseline'

ckpt = os.path.join(T1_OUT_DIR, 'ckpt.pt')
INIT_T1 = 'resume' if os.path.exists(ckpt) else 'scratch'
print(f"Checkpoint : {'found — resuming' if INIT_T1 == 'resume' else 'not found — training from scratch'}")
print(f"Config     : {T1_CONFIG}")
print(f"init_from  : {INIT_T1}")

Checkpoint : not found — training from scratch
Config     : config/train_t1_baseline.py
init_from  : scratch


In [ ]:
import os
os.chdir(LOCAL_DIR)

print(f"  Config   : {T1_CONFIG}")
print(f"  Out dir  : {T1_OUT_DIR}")
print(f"  Init     : {INIT_T1}  ({'resuming from checkpoint' if INIT_T1 == 'resume' else 'training from scratch'})")
print(f"  Params   : ~30.2M  (≤ 32M ✓)")
print(f"  Steps    : 10,000  (~88 epochs over ROCStories)")
print(f"  ETA      : ~8–10 min on A100\n")

rc = run_streaming(
    ['python', '-u', 'train.py', T1_CONFIG, f'--init_from={INIT_T1}'],
    label=f"Task 1 Baseline Training  ·  6L/6H/384D vanilla nanoGPT"
)

if rc == 0:
    print("\n✓  Training complete.  Checkpoint saved to:", T1_OUT_DIR)
else:
    print(f"\n⚠  Process exited with code {rc}.")
    print("   Re-run this cell — it will resume from the last checkpoint automatically.")

  Config   : config/train_t1_baseline.py
  Out dir  : out-t1-baseline
  Init     : scratch  (training from scratch)
  Params   : ~30.2M  (≤ 32M ✓)
  Steps    : 10,000  (~88 epochs over ROCStories)
  ETA      : ~8–10 min on A100


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ▶  Task 1 Baseline Training  ·  6L/6H/384D vanilla nanoGPT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Overriding config with config/train_t1_baseline.py:
# ============================================================
# config/train_t1_baseline.py
#
# TASK 1 — Official Baseline nanoGPT on ROCStories
#
# Follows the official nanoGPT "baby GPT" configuration:
#   https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py
#
# Prof constraint: n_layer=6, n_head=6, n_embd=384  ≈ 30.2M params
#                 (must not exceed 32M total params)
#
# Architecture: Vanilla nanoGPT (learned PE, LayerNorm, GELU MLP)
# No modern improvements — this is the required Ta

### 1.4 Evaluation

**Perplexity (PPL)** is the primary quantitative metric:  
$$\text{PPL} = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i \mid w_{<i})\right)$$

Lower PPL = model assigns higher probability to the true next tokens.

**Passing threshold (prof announcement):** PPL ≤ 25.0 on the provided 19,633-story test set.

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

print("=" * 55)
print("TASK 1 — Perplexity on ROCStories test set")
print("=" * 55)
subprocess.run([
    'python', 'eval.py',
    '--init_from=resume',
    f'--out_dir={T1_OUT_DIR}',
    '--input_file=data/rocstories/eval_stories.txt'
], check=False)

print()
print("=" * 55)
print("TASK 1 — Qualitative story samples")
print("=" * 55)
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume',
    f'--out_dir={T1_OUT_DIR}',
    '--start=FILE:data/rocstories/eval_prompts.txt',
    '--batch_prompts=True',
    '--max_new_tokens=150',
    f'--output_file={T1_OUT_DIR}/generated_stories.jsonl'
], check=False)

TASK 1 — Perplexity on ROCStories test set

TASK 1 — Qualitative story samples


CompletedProcess(args=['python', 'sample_batch.py', '--init_from=resume', '--out_dir=out-t1-baseline', '--start=FILE:data/rocstories/eval_prompts.txt', '--batch_prompts=True', '--max_new_tokens=150', '--output_file=out-t1-baseline/generated_stories.jsonl'], returncode=0)

---
## §2 · Task 2 — Exploration

### Research Questions

We investigate two complementary directions within the 32 M parameter budget:

**Direction 1 — Architecture ablation** (Cells 2.1–2.3):  
Which modern architectural components (RoPE, RMSNorm+SwiGLU, QK-Norm) individually
benefit story generation at the sub-32 M scale?

**Direction 2 — Mixed instruction training** (Cells 2.4–2.5):  
Can mixing instruction-prefixed stories into training improve narrative quality
without degrading perplexity on plain continuation prompts?
This is motivated by InstructGPT (Ouyang et al., 2022) but studied
here at a scale and domain (short narrative generation) that has not
been previously reported.

---
### 2.1 Dataset Preparation

Four variants are prepared:

| Dataset | Stories | Tokens | Purpose |
|---------|---------|--------|---------|
| `rocstories` | 78,528 | 3.7 M | Tasks 1 + ablations |
| `rocstories_structured` | 78,528 | 7.0 M | Structured-format ablation |
| `tinystories` | 500,000 | 107 M | Optional pretraining data |
| `mixed` | 78,528 × 3 formats | ~6.7 M | Instruction experiment |

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

# 1. Structured format (already done if Task 1 ran — skip if bin exists)
if not os.path.exists('data/rocstories_structured/train.bin'):
    print("Preparing ROCStories structured format...")
    !python data/rocstories/prepare.py --structured
else:
    print("Structured dataset already exists — skipping")

# 2. TinyStories (may take ~5 min on first download)
if not os.path.exists('data/tinystories/train.bin'):
    print("\nPreparing TinyStories (~5 min first time)...")
    !python data/tinystories/prepare.py
else:
    print("TinyStories already exists — skipping")

# 3. Mixed instruction dataset
if not os.path.exists('data/mixed/train.bin'):
    print("\nBuilding mixed instruction dataset...")
    !python data/mixed/prepare.py
else:
    print("Mixed dataset already exists — skipping")

# Summary
print("\n" + "=" * 50)
print("Dataset Summary")
print("=" * 50)
for ds, split in [('rocstories','train'), ('rocstories','val'),
                   ('rocstories_structured','train'),
                   ('tinystories','train'), ('mixed','train'), ('mixed','val')]:
    p = f'data/{ds}/{split}.bin'
    if os.path.exists(p):
        n = len(np.fromfile(p, dtype=np.uint16))
        print(f"  {ds}/{split}.bin: {n/1e6:.2f}M tokens")

Preparing ROCStories structured format...
[prepare] Trying to load: mintujupally/ROCStories ...
Repo card metadata block was not found. Setting CardData to empty.
[prepare] Columns: ['text']
[prepare] Loaded 78,528 stories from mintujupally/ROCStories (structured format)
[prepare] Split: 70,676 train | 7,852 val

[prepare] Sample story:
<story>
<s1>The boy went to a video arcade.</s1>
<s2>He played his favorite machine.</s2>
<s3>His games didn't go very well.</s3>
<s4>He told the owner about his experience.</s4>
<s5>The owner explained that he had made the game settings harder.</s5>
</story>

[prepare] train.bin: 7,043,566 tokens  (avg 99.7 tok/story)  -> /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code-v2/data/rocstories_structured/train.bin
[prepare] val.bin: 781,392 tokens  (avg 99.5 tok/story)  -> /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code-v2/data/rocstories_structure

### 2.2 Architecture Ablation Design

We isolate **three orthogonal architectural modifications** from the vanilla nanoGPT:

| Config | Changes from vanilla | Out-dir |
|--------|---------------------|----------|
| A — Vanilla | none (baseline) | `out-t2-vanilla` |
| B — +RoPE | Rotary positional embeddings (Su et al., 2022) | `out-t2-rope` |
| C — +RMSNorm+SwiGLU | RMS layer norm + gated FFN (Zhang & Sennrich, 2019; Shazeer, 2020) | `out-t2-ffn` |
| D — +QK-Norm ★ | Query-Key normalisation (Henry et al., 2020; Gemma 2, 2024) | `out-t2-qknorm` |
| E — All Modern | All three combined | `out-t2-all-modern` |

★ **Novel contribution:** QK-Norm's effect on sub-32 M narrative-generation models is not previously reported.

All experiments use identical hyperparameters (6L/6H/384D, 10K steps, lr=6e-4) for a controlled comparison.
Each model is ~30.2 M parameters, well within the 32 M constraint.

In [ ]:
import os, subprocess, json, time
os.chdir(LOCAL_DIR)

ABLATIONS = [
    ('config/train_t2_ablation_a.py', 'out-t2-vanilla',    'A. Vanilla'),
    ('config/train_t2_ablation_b.py', 'out-t2-rope',       'B. +RoPE'),
    ('config/train_t2_ablation_c.py', 'out-t2-ffn',        'C. +RMSNorm+SwiGLU'),
    ('config/train_t2_ablation_d.py', 'out-t2-qknorm',     'D. +QK-Norm (novel)'),
    ('config/train_t2_ablation_e.py', 'out-t2-all-modern', 'E. All Modern'),
]

ppl_results = {}
train_times  = {}
start_all   = time.time()
total       = len(ABLATIONS)

print(f"{'█' * 62}")
print(f"  TASK 2 — ARCHITECTURE ABLATION STUDY")
print(f"  {total} experiments  ·  ~30.2M params each  ·  10K steps each")
print(f"  Estimated total: ~{total * 10}-{total * 12} min on A100")
print(f"{'█' * 62}")

for i, (cfg, out_dir, label) in enumerate(ABLATIONS, 1):
    ckpt = os.path.join(out_dir, 'ckpt.pt')
    init = 'resume' if os.path.exists(ckpt) else 'scratch'
    elapsed_so_far = (time.time() - start_all) / 60

    print(f"\n\n{'▓' * 62}")
    print(f"  Experiment {i}/{total}  :  {label}")
    print(f"  Config          :  {cfg}")
    print(f"  init_from       :  {init}  ({'resuming' if init == 'resume' else 'scratch'})")
    print(f"  Total elapsed   :  {elapsed_so_far:.1f} min")
    print(f"{'▓' * 62}")

    # ── Train ──────────────────────────────────────────────────────────
    t0 = time.time()
    run_streaming(
        ['python', '-u', 'train.py', cfg, f'--init_from={init}'],
        label=f"[{i}/{total}] Training  {label}"
    )
    train_time = (time.time() - t0) / 60
    train_times[label] = train_time

    # ── Eval PPL ────────────────────────────────────────────────────────
    print(f"\n  ── Evaluating PPL: {label} ──")
    r = subprocess.run(
        ['python', 'eval.py', '--init_from=resume', f'--out_dir={out_dir}',
         '--input_file=data/rocstories/eval_stories.txt'],
        capture_output=True, text=True, check=False
    )
    for line in r.stdout.strip().split('\n'):
        print(f"  {line}")
    if r.returncode != 0 and r.stderr:
        print(f"  STDERR: {r.stderr[:300]}")
    for line in r.stdout.split('\n'):
        if 'ppl' in line.lower() or 'perplexity' in line.lower():
            ppl_results[label] = line.strip()

    # ── Generate samples ───────────────────────────────────────────────
    print(f"\n  ── Generating story samples: {label} ──")
    subprocess.run([
        'python', 'sample_batch.py',
        '--init_from=resume', f'--out_dir={out_dir}',
        '--start=FILE:data/rocstories/eval_prompts.txt',
        '--batch_prompts=True', '--max_new_tokens=150',
        f'--output_file={out_dir}/generated_stories.jsonl'
    ], check=False)

    print(f"\n  ✓  Experiment {i}/{total} done  —  train: {train_time:.1f} min")

# ── Final summary ──────────────────────────────────────────────────────────
total_time = (time.time() - start_all) / 60
print(f"\n\n{'═' * 62}")
print(f"  ABLATION STUDY COMPLETE  —  {total_time:.1f} min total")
print(f"{'═' * 62}")
print(f"  {'Config':28s}  {'Train (min)':>11s}  {'PPL Result'}")
print(f"  {'─' * 60}")
for lbl, train_t in train_times.items():
    ppl_line = ppl_results.get(lbl, '(not captured)')
    print(f"  {lbl:28s}  {train_t:>10.1f}  {ppl_line}")
print(f"\n  Train logs: each out-dir contains train_log.jsonl")
print(f"  Plots     : run the Analysis cell (§2.4) to generate ablation_curves.png")

██████████████████████████████████████████████████████████████
  TASK 2 — ARCHITECTURE ABLATION STUDY
  5 experiments  ·  ~30.2M params each  ·  10K steps each
  Estimated total: ~50-60 min on A100
██████████████████████████████████████████████████████████████


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  Experiment 1/5  :  A. Vanilla
  Config          :  config/train_t2_ablation_a.py
  init_from       :  scratch  (scratch)
  Total elapsed   :  0.0 min
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ▶  [1/5] Training  A. Vanilla
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Overriding config with config/train_t2_ablation_a.py:
# ============================================================
# config/train_t2_ablation_a.py
#
# TASK 2 — Ablation A: Vanilla GPT (reference point)
#
# Same architecture as Task 1 baseline but trained for
# 10,000 steps to match the other abla

<IPython.core.display.Javascript object>

step 0: train loss 10.8743, val loss 10.8741
iter 0: loss 10.8810, time 38481.65ms, mfu -100.00%
iter 10: loss 10.1066, time 278.21ms, mfu 7.05%
iter 20: loss 9.7495, time 279.31ms, mfu 7.05%
iter 30: loss 9.3328, time 279.83ms, mfu 7.04%
iter 40: loss 8.8208, time 279.53ms, mfu 7.04%
iter 50: loss 8.2380, time 279.60ms, mfu 7.04%
iter 60: loss 7.5984, time 279.71ms, mfu 7.03%
iter 70: loss 7.0738, time 279.83ms, mfu 7.03%
iter 80: loss 6.7593, time 282.77ms, mfu 7.02%
iter 90: loss 6.4251, time 281.43ms, mfu 7.02%


<IPython.core.display.Javascript object>

iter 100: loss 6.2980, time 281.45ms, mfu 7.01%
iter 110: loss 6.0941, time 282.79ms, mfu 7.00%
iter 120: loss 5.9560, time 281.75ms, mfu 7.00%
iter 130: loss 5.9244, time 281.01ms, mfu 7.00%
iter 140: loss 5.8744, time 283.03ms, mfu 6.99%
iter 150: loss 5.8330, time 280.93ms, mfu 6.99%
iter 160: loss 5.7974, time 283.46ms, mfu 6.98%
iter 170: loss 5.7293, time 282.99ms, mfu 6.98%
iter 180: loss 5.7032, time 283.46ms, mfu 6.97%
iter 190: loss 5.6659, time 284.36ms, mfu 6.96%
iter 200: loss 5.6626, time 284.47ms, mfu 6.96%
iter 210: loss 5.6675, time 282.28ms, mfu 6.96%
iter 220: loss 5.6352, time 284.32ms, mfu 6.95%
iter 230: loss 5.5015, time 283.28ms, mfu 6.95%
iter 240: loss 5.5796, time 286.39ms, mfu 6.94%
step 250: train loss 4.6383, val loss 4.6872
saving checkpoint to out-t2-vanilla/ckpt.pt
iter 250: loss 5.5317, time 5391.94ms, mfu 6.28%
iter 260: loss 5.4914, time 284.61ms, mfu 6.34%


...

iter 9760: loss 2.7520, time 324.62ms, mfu 6.03%
iter 9770: loss 2.7684, time 327.64ms, mfu 6.09%
iter 9780: loss 2.7478, time 329.12ms, mfu 6.15%
iter 9790: loss 2.7225, time 325.10ms, mfu 6.20%
iter 9800: loss 2.7665, time 328.41ms, mfu 6.25%
iter 9810: loss 2.7754, time 328.79ms, mfu 6.29%
iter 9820: loss 2.7525, time 328.00ms, mfu 6.33%
iter 9830: loss 2.7541, time 328.91ms, mfu 6.36%
iter 9840: loss 2.7474, time 333.41ms, mfu 6.38%
iter 9850: loss 2.7610, time 333.79ms, mfu 6.39%
iter 9860: loss 2.7847, time 326.25ms, mfu 6.42%
iter 9870: loss 2.7475, time 330.72ms, mfu 6.44%
iter 9880: loss 2.8008, time 324.84ms, mfu 6.47%
iter 9890: loss 2.6988, time 327.96ms, mfu 6.49%


<IPython.core.display.Javascript object>

iter 9900: loss 2.7397, time 328.32ms, mfu 6.50%
iter 9910: loss 2.7403, time 330.64ms, mfu 6.51%
iter 9920: loss 2.7400, time 329.47ms, mfu 6.53%
iter 9930: loss 2.7485, time 327.96ms, mfu 6.54%
iter 9940: loss 2.7556, time 331.16ms, mfu 6.55%
iter 9950: loss 2.7795, time 327.98ms, mfu 6.56%
iter 9960: loss 2.7672, time 326.75ms, mfu 6.57%
iter 9970: loss 2.7200, time 330.05ms, mfu 6.57%
iter 9980: loss 2.7358, time 332.08ms, mfu 6.57%
iter 9990: loss 2.7259, time 329.28ms, mfu 6.58%
step 10000: train loss 0.9171, val loss 4.0570
saving checkpoint to out-t2-all-modern/ckpt.pt
iter 10000: loss 2.7432, time 6551.28ms, mfu 5.96%
Training complete — saving final checkpoint
saving checkpoint to out-t2-all-modern/ckpt_final.pt
saving checkpoint to out-t2-all-modern/ckpt.pt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓  finished in 65m 7s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── Evaluating PPL: E. All Modern ──
  Overriding: init_from = resum

<IPython.core.display.Javascript object>

  [keep-alive] 14:39:43
  ✓  Experiment 5/5 done  —  train: 65.1 min


══════════════════════════════════════════════════════════════
  ABLATION STUDY COMPLETE  —  306.9 min total
══════════════════════════════════════════════════════════════
  Config                        Train (min)  PPL Result
  ────────────────────────────────────────────────────────────
  A. Vanilla                          58.2  ppl             : 34.69
  B. +RoPE                            59.1  ppl             : 32.10
  C. +RMSNorm+SwiGLU                  63.3  ppl             : 40.47
  D. +QK-Norm (novel)                 59.4  ppl             : 31.88
  E. All Modern                       65.1  ppl             : 42.26

  Train logs: each out-dir contains train_log.jsonl
  Plots     : run the Analysis cell (§2.4) to generate ablation_curves.png


### 2.3 Novel Experiment — Mixed Instruction + Continuation Training

**Motivation:** Modern instruction-tuned LLMs (InstructGPT, LLaMA-chat) demonstrate that training
on both instruction prompts and raw continuations enables a single model to serve multiple
interaction modes. For small story-generation models, we hypothesise that instruction exposure
can improve narrative coherence without degrading PPL on plain continuation.

**Training mixture** (`data/mixed/train.bin`):

| Format | Proportion | Example |
|--------|-----------|----------|
| A — Plain continuation | 55% | `The boy went to a video arcade...` |
| B — Instruction-prefixed | 30% | `Write a story about: the boy went to a video arcade.\n<story>` |
| C — Structured XML | 15% | `<story><s1>...</s1>...<s5>...</s5></story>` |

The **validation set is plain ROCStories only**, ensuring PPL comparisons are directly comparable
to Task 1 and the ablation baselines.

**Reference:** Ouyang, L. et al. (2022). Training language models to follow instructions with
human feedback. arXiv:2203.02155.

In [ ]:
import os
os.chdir(LOCAL_DIR)

T3_CONFIG  = 'config/train_t3_best.py'
T3_OUT_DIR = 'out-t3-best'

ckpt = os.path.join(T3_OUT_DIR, 'ckpt.pt')
init_t3 = 'resume' if os.path.exists(ckpt) else 'scratch'

print(f"  Model    : All-Modern 30M  (RoPE + RMSNorm + SwiGLU + QK-Norm)")
print(f"  Dataset  : data/mixed/  (55% plain · 30% instruction · 15% structured)")
print(f"  Config   : {T3_CONFIG}")
print(f"  Out dir  : {T3_OUT_DIR}")
print(f"  Init     : {init_t3}  ({'resuming from checkpoint' if init_t3 == 'resume' else 'training from scratch'})")
print(f"  Steps    : 15,000  (longer than ablations for best PPL)")
print(f"  ETA      : ~12–15 min on A100\n")
print(f"  NOTE: this model is also the Task 3 HuggingFace submission.")

rc = run_streaming(
    ['python', '-u', 'train.py', T3_CONFIG, f'--init_from={init_t3}'],
    label="Task 3 / Instruction Experiment  ·  All-Modern 30M + Mixed Data"
)

if rc == 0:
    print(f"\n✓  Training complete.  Checkpoint saved to: {T3_OUT_DIR}")
else:
    print(f"\n⚠  Process exited with code {rc}.")
    print("   Re-run this cell — it will resume from the last checkpoint automatically.")

  Model    : All-Modern 30M  (RoPE + RMSNorm + SwiGLU + QK-Norm)
  Dataset  : data/mixed/  (55% plain · 30% instruction · 15% structured)
  Config   : config/train_t3_best.py
  Out dir  : out-t3-best
  Init     : scratch  (training from scratch)
  Steps    : 15,000  (longer than ablations for best PPL)
  ETA      : ~12–15 min on A100

  NOTE: this model is also the Task 3 HuggingFace submission.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ▶  Task 3 / Instruction Experiment  ·  All-Modern 30M + Mixed Data
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Overriding config with config/train_t3_best.py:
# ============================================================
# config/train_t3_best.py
#
# TASK 3 — Best Checkpoint Submission (≤ 32M params)
#
# Architecture: All-Modern 30M (RoPE + RMSNorm + SwiGLU + QK-Norm)
#   → Best configuration identified in the Task 2 ablation study
#
# Data: Mixed instruction dataset (data/mixed/)
#   55% plain continuation (

<IPython.core.display.Javascript object>

step 0: train loss 10.8943, val loss 10.8926
W0318 14:47:42.248000 99269 torch/_inductor/utils.py:1679] [0/1_1] Not enough SMs to use max_autotune_gemm mode
iter 0: loss 10.8917, time 47132.71ms, mfu -100.00%


<IPython.core.display.Javascript object>

iter 10: loss 10.5918, time 312.36ms, mfu 6.99%
iter 20: loss 10.1392, time 316.61ms, mfu 6.98%
iter 30: loss 9.7653, time 319.16ms, mfu 6.97%
iter 40: loss 9.1375, time 318.79ms, mfu 6.96%
iter 50: loss 8.9296, time 316.54ms, mfu 6.95%
iter 60: loss 8.5094, time 319.18ms, mfu 6.94%
iter 70: loss 8.1625, time 313.18ms, mfu 6.94%
iter 80: loss 7.5877, time 319.18ms, mfu 6.93%
iter 90: loss 7.2163, time 319.36ms, mfu 6.92%
iter 100: loss 6.7741, time 319.83ms, mfu 6.91%
iter 110: loss 6.5896, time 319.25ms, mfu 6.91%
iter 120: loss 5.9906, time 319.07ms, mfu 6.90%
iter 130: loss 5.8111, time 321.22ms, mfu 6.89%
iter 140: loss 5.6599, time 321.18ms, mfu 6.88%
iter 150: loss 5.4435, time 321.14ms, mfu 6.87%
iter 160: loss 5.5477, time 320.32ms, mfu 6.87%
iter 170: loss 5.2795, time 319.51ms, mfu 6.86%


...

iter 14560: loss 2.2646, time 324.70ms, mfu 6.34%
iter 14570: loss 2.2631, time 322.65ms, mfu 6.39%
iter 14580: loss 2.2669, time 322.60ms, mfu 6.42%
iter 14590: loss 2.2553, time 331.34ms, mfu 6.44%
iter 14600: loss 2.2703, time 326.30ms, mfu 6.47%
iter 14610: loss 2.2788, time 323.44ms, mfu 6.49%
iter 14620: loss 2.2881, time 326.09ms, mfu 6.51%
iter 14630: loss 2.2422, time 332.09ms, mfu 6.52%
iter 14640: loss 2.2556, time 325.93ms, mfu 6.54%
iter 14650: loss 2.2694, time 329.99ms, mfu 6.55%
iter 14660: loss 2.2368, time 326.54ms, mfu 6.56%
iter 14670: loss 2.2656, time 330.60ms, mfu 6.57%
iter 14680: loss 2.2799, time 331.10ms, mfu 6.57%
iter 14690: loss 2.2497, time 331.68ms, mfu 6.57%
iter 14700: loss 2.2367, time 331.49ms, mfu 6.57%
iter 14710: loss 2.2630, time 331.67ms, mfu 6.57%
iter 14720: loss 2.2494, time 331.59ms, mfu 6.57%


<IPython.core.display.Javascript object>

iter 14730: loss 2.2456, time 331.08ms, mfu 6.58%
iter 14740: loss 2.2064, time 331.51ms, mfu 6.58%
iter 14750: loss 2.2119, time 328.41ms, mfu 6.59%
iter 14760: loss 2.2181, time 330.52ms, mfu 6.59%
iter 14770: loss 2.2270, time 333.40ms, mfu 6.58%
iter 14780: loss 2.2614, time 333.45ms, mfu 6.58%
iter 14790: loss 2.1910, time 332.50ms, mfu 6.58%
iter 14800: loss 2.2704, time 335.02ms, mfu 6.57%
iter 14810: loss 2.3035, time 331.87ms, mfu 6.57%
iter 14820: loss 2.1999, time 331.75ms, mfu 6.57%
iter 14830: loss 2.2850, time 330.61ms, mfu 6.58%
iter 14840: loss 2.2308, time 329.60ms, mfu 6.58%
iter 14850: loss 2.2905, time 330.73ms, mfu 6.58%
iter 14860: loss 2.2369, time 328.11ms, mfu 6.59%
iter 14870: loss 2.2732, time 325.06ms, mfu 6.60%
iter 14880: loss 2.2631, time 329.60ms, mfu 6.61%


<IPython.core.display.Javascript object>

iter 14890: loss 2.2369, time 323.12ms, mfu 6.62%
iter 14900: loss 2.2596, time 328.22ms, mfu 6.63%
iter 14910: loss 2.2808, time 326.31ms, mfu 6.63%
iter 14920: loss 2.2635, time 330.01ms, mfu 6.63%
iter 14930: loss 2.2728, time 328.74ms, mfu 6.63%
iter 14940: loss 2.2489, time 328.88ms, mfu 6.63%
iter 14950: loss 2.2632, time 329.49ms, mfu 6.63%
iter 14960: loss 2.2738, time 334.26ms, mfu 6.62%
iter 14970: loss 2.2292, time 330.27ms, mfu 6.62%
iter 14980: loss 2.2451, time 330.19ms, mfu 6.62%
iter 14990: loss 2.2429, time 332.49ms, mfu 6.62%
step 15000: train loss 0.5490, val loss 0.9115
saving checkpoint to out-t3-best/ckpt.pt
iter 15000: loss 2.2291, time 11365.44ms, mfu 5.97%
Training complete — saving final checkpoint
saving checkpoint to out-t3-best/ckpt_final.pt
saving checkpoint to out-t3-best/ckpt.pt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓  finished in 92m 43s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✓  Training complete.  Ch

### 2.4 Analysis and Visualisation

The following cell generates:
1. **Learning-curve overlay** — val loss across all 5 architecture ablations
2. **Bar chart** — final val loss per experiment
3. **Story quality comparison** — lexical diversity and avg length per model

All outputs are saved as `.png` files for the report.

In [ ]:
import os, json, subprocess
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.chdir(LOCAL_DIR)

# ── Discover completed experiments ───────────────────────────────────────────
ALL_EXPS = [
    ('out-t1-baseline',   'Task 1 Baseline'),
    ('out-t2-vanilla',    'A. Vanilla'),
    ('out-t2-rope',       'B. +RoPE'),
    ('out-t2-ffn',        'C. +RMSNorm+SwiGLU'),
    ('out-t2-qknorm',     'D. +QK-Norm'),
    ('out-t2-all-modern', 'E. All Modern'),
    ('out-t3-best',       'F. +Mixed Instr.'),
]

available = [
    (d, l, os.path.join(d, 'train_log.jsonl'))
    for d, l in ALL_EXPS
    if os.path.exists(os.path.join(d, 'train_log.jsonl'))
]

print(f"Found {len(available)}/{len(ALL_EXPS)} completed experiments:")
for d, l, p in available:
    print(f"  \u2713 {l:28s} — {p}")
missing = [(d, l) for d, l in ALL_EXPS
           if not os.path.exists(os.path.join(d, 'train_log.jsonl'))]
for d, l in missing:
    print(f"  \u2717 {l:28s} — not yet run")

if not available:
    print("\nNo completed experiments yet. Run the training cells first.")
else:
    COLORS = ['#555555','#E53935','#1E88E5','#43A047','#FB8C00','#8E24AA','#00ACC1']

    # ── 1. Learning-curve overlay ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    for (d, label, log_path), color in zip(available, COLORS):
        steps, val_losses = [], []
        with open(log_path) as f:
            for line in f:
                entry = json.loads(line)
                if entry.get('val_loss') is not None:
                    steps.append(entry['step'])
                    val_losses.append(entry['val_loss'])
        if steps:
            ax.plot(steps, val_losses, label=label, color=color, linewidth=1.8)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Validation Loss', fontsize=12)
    ax.set_title('Task 2: Validation Loss — Architecture Ablation', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('ablation_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\n\u2713 Saved: ablation_curves.png")

    # ── 2. Bar chart of final val loss ─────────────────────────────────────
    bar_labels, bar_vals = [], []
    for d, label, log_path in available:
        with open(log_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in reversed(lines):
            entry = json.loads(line)
            if entry.get('val_loss') is not None:
                bar_labels.append(label)
                bar_vals.append(entry['val_loss'])
                break

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(bar_labels, bar_vals, color=COLORS[:len(bar_vals)],
                  edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, bar_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Final Validation Loss', fontsize=12)
    ax.set_title('Task 2: Final Validation Loss by Configuration', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20, ha='right', fontsize=9)
    plt.tight_layout()
    plt.savefig('ablation_bar_chart.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\u2713 Saved: ablation_bar_chart.png")

    # ── 3. Summary table ──────────────────────────────────────────────────
    print(f"\n  {'Config':30s} {'Final Val Loss':>15s}")
    print(f"  {'─'*47}")
    for lbl, val in sorted(zip(bar_labels, bar_vals), key=lambda x: x[1]):
        print(f"  {lbl:30s} {val:15.4f}")

    # ── 4. Story quality (lexical diversity) ─────────────────────────────
    print(f"\n  {'Config':30s} {'Stories':>8s} {'Avg Len':>8s} {'Type/Token':>12s}")
    print(f"  {'─'*60}")
    for d, label, _ in available:
        stories_path = os.path.join(d, 'generated_stories.jsonl')
        if not os.path.exists(stories_path):
            continue
        texts = []
        with open(stories_path) as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    texts.append(obj.get('generated', obj.get('text', '')))
                except json.JSONDecodeError:
                    pass
        if not texts:
            continue
        all_words = ' '.join(texts).lower().split()
        ttr = len(set(all_words)) / max(len(all_words), 1)
        avg_len = sum(len(t.split()) for t in texts) / len(texts)
        print(f"  {label:30s} {len(texts):8d} {avg_len:8.1f} {ttr:12.4f}")

print(f"\n  Download ablation_curves.png and ablation_bar_chart.png for your report.")

Found 7/7 completed experiments:
  ✓ Task 1 Baseline              — out-t1-baseline/train_log.jsonl
  ✓ A. Vanilla                   — out-t2-vanilla/train_log.jsonl
  ✓ B. +RoPE                     — out-t2-rope/train_log.jsonl
  ✓ C. +RMSNorm+SwiGLU           — out-t2-ffn/train_log.jsonl
  ✓ D. +QK-Norm                  — out-t2-qknorm/train_log.jsonl
  ✓ E. All Modern                — out-t2-all-modern/train_log.jsonl
  ✓ F. +Mixed Instr.             — out-t3-best/train_log.jsonl

✓ Saved: ablation_curves.png
✓ Saved: ablation_bar_chart.png

  Config                          Final Val Loss
  ───────────────────────────────────────────────
  F. +Mixed Instr.                        0.9115
  D. +QK-Norm                             3.7290
  Task 1 Baseline                         3.7578
  A. Vanilla                              3.7621
  B. +RoPE                                3.8294
  C. +RMSNorm+SwiGLU                      3.9801
  E. All Modern                           4.0570

  Conf

In [5]:
import json

with open('out-t3-best/generated_stories.jsonl') as f:
    for i, line in enumerate(f):
        print(json.loads(line))
        if i == 2:
            break

{'prompt': 'Emily forgot her umbrella before leaving for work.', 'generated_text': 'Emily forgot her umbrella before leaving for work. She got dressed and made it to work. When she arrived at work, her dog jumped on the scale and made a loud thump. Her umbrella was almost unusable! Now Mary does not have to wear out to walk to school during the day.', 'params': {'max_new_tokens': 200, 'temperature': 0.75, 'top_k': 50, 'top_p': 0.9, 'repetition_penalty': 1.2}}
{'prompt': 'Tom decided to cook dinner for his friends.', 'generated_text': 'Tom decided to cook dinner for his friends. He decided he would make pasta for all his wife. He bought all the ingredients. Everyone at work loved the dish. Tom had a great time cooking it up!', 'params': {'max_new_tokens': 200, 'temperature': 0.75, 'top_k': 50, 'top_p': 0.9, 'repetition_penalty': 1.2}}
{'prompt': 'Lily wanted to start jogging every morning.', 'generated_text': 'Lily wanted to start jogging every morning. She put off working two hours eve

---
## §3 · Task 3 — Best Checkpoint Submission

The best submission model is the **all-modern 30 M + mixed instruction training** model already
trained in §2.3 (`out-t3-best/ckpt.pt`).  No new training is needed here.

### 3.1 Final Evaluation

Comprehensive evaluation on the full public test set.

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

print("=" * 55)
print("TASK 3 — Final PPL on test set")
print("=" * 55)
subprocess.run([
    'python', 'eval.py',
    '--init_from=resume',
    '--out_dir=out-t3-best',
    '--input_file=data/rocstories/eval_stories.txt'
], check=False)

print()
print("=" * 55)
print("TASK 3 — Story generation (plain prompts)")
print("=" * 55)
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume', '--out_dir=out-t3-best',
    '--start=FILE:data/rocstories/eval_prompts.txt',
    '--batch_prompts=True', '--max_new_tokens=200',
    '--output_file=out-t3-best/generated_stories.jsonl'
], check=False)

print()
print("=" * 55)
print("TASK 3 — Story generation (instruction prompts)")
print("=" * 55)
# Also test instruction-mode generation to verify dual capability
instr_prompts = [
    'Write a story about: a girl who lost her dog.\n',
    'Tell me a 5-sentence story involving a birthday surprise.\n',
    'Generate a brief narrative about learning to ride a bike.\n',
]
import json as _json
with open('/tmp/instr_prompts.txt', 'w') as f:
    f.write('\n---\n'.join(instr_prompts))
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume', '--out_dir=out-t3-best',
    '--start=FILE:/tmp/instr_prompts.txt',
    '--batch_prompts=True', '--max_new_tokens=200',
    '--output_file=out-t3-best/generated_instruction_stories.jsonl'
], check=False)

TASK 3 — Final PPL on test set


<IPython.core.display.Javascript object>

  [keep-alive] 16:19:38
TASK 3 — Story generation (plain prompts)

TASK 3 — Story generation (instruction prompts)


CompletedProcess(args=['python', 'sample_batch.py', '--init_from=resume', '--out_dir=out-t3-best', '--start=FILE:/tmp/instr_prompts.txt', '--batch_prompts=True', '--max_new_tokens=200', '--output_file=out-t3-best/generated_instruction_stories.jsonl'], returncode=1)

### 3.2 Package and Upload to HuggingFace

Files uploaded per assignment specification (`hf_load.py`):
- `ckpt.pt` — final checkpoint
- `model.py` — modified architecture (with RoPE, RMSNorm, SwiGLU, QK-Norm)
- `sample_params.json` — generation hyperparameters used for sampling
- `eval.py` + `configurator.py` — for evaluator to load and run the model

**Submission repo:** `<HF_USERNAME>/nanoGPT_hw`

In [ ]:
# ── §3.4 HuggingFace Submission ───────────────────────────────────────────
import os, shutil, json, math, torch, subprocess
from google.colab import userdata
os.chdir(LOCAL_DIR)

HF_USERNAME = "YOUR_HF_USERNAME"   # ← REPLACE
HF_TOKEN    = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

T3_DIR     = 'out-t3-best'
SUBMIT_DIR = 'submission_hf'
os.makedirs(SUBMIT_DIR, exist_ok=True)

# Use best checkpoint (not final — avoids the overfitting trap)
best_ckpt  = os.path.join(T3_DIR, 'ckpt_best.pt')
final_ckpt = os.path.join(T3_DIR, 'ckpt.pt')
src_ckpt   = best_ckpt if os.path.exists(best_ckpt) else final_ckpt
print(f"Using: {src_ckpt}")

# Verify before submitting
ck       = torch.load(src_ckpt, map_location='cpu', weights_only=False)
best_ppl = math.exp(ck['best_val_loss'])
from model import GPT, GPTConfig
m = GPT(GPTConfig(**ck['model_args']))
n = sum(p.numel() for p in m.parameters())
print(f"  Best val PPL: {best_ppl:.1f}")
print(f"  Parameters:   {n/1e6:.2f}M {'✓' if n <= 32e6 else '✗ OVER 32M LIMIT'}")
assert n <= 32_000_000, "Model exceeds 32M param limit!"
assert best_ppl < 40, f"PPL {best_ppl:.1f} seems too high — check dataset and checkpoint"

# Write generation params
params = {"temperature": 0.75, "top_k": 50, "top_p": 0.9, "repetition_penalty": 1.2}
with open('sample_params.json', 'w') as f:
    json.dump(params, f, indent=2)

# Copy all required files
for src, dst in [
    (src_ckpt,            f'{SUBMIT_DIR}/ckpt.pt'),   # must be named ckpt.pt
    ('model.py',          f'{SUBMIT_DIR}/model.py'),
    ('sample_params.json',f'{SUBMIT_DIR}/sample_params.json'),
    ('eval.py',           f'{SUBMIT_DIR}/eval.py'),
    ('configurator.py',   f'{SUBMIT_DIR}/configurator.py'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)

print("\nSubmission folder contents:")
for fname in sorted(os.listdir(SUBMIT_DIR)):
    mb = os.path.getsize(f'{SUBMIT_DIR}/{fname}') / 1e6
    print(f"  {fname:<30s} {mb:.1f} MB")

REPO_ID = f"{HF_USERNAME}/nanoGPT_hw"
result = subprocess.run([
    'python', 'hf_load.py', 'upload',
    '--local-dir', SUBMIT_DIR,
    '--repo-id',   REPO_ID,
    '--token',     HF_TOKEN,
    '--commit-message', f'Best nanoGPT checkpoint — {n/1e6:.1f}M params, best PPL {best_ppl:.1f}'
], check=False)

print(f"\n✓ Uploaded to: https://huggingface.co/{REPO_ID}")
print(f"  Canvas submission: {REPO_ID}")

---
## §4 · Task 4 — Arena Competition (Optional, 152 M)

> **Note:** This section is entirely optional and only relevant for the **arena competition**.
> The 152 M model **must NOT be submitted** to HuggingFace for Task 3 grading — it exceeds the 32 M constraint.

The arena model uses the same LLaMA-style architecture scaled to 152 M (12L/12H/768D),
trained on the combined ROCStories + TinyStories dataset (~110 M tokens) for richer language capacity.

### Generation strategy for human judging:
- Temperature: 0.85 (slightly higher for more creative, varied stories)
- Top-K: 50, Top-P: 0.9
- Repetition penalty: 1.2 (prevents looping, critical for coherence)

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

# Prepare combined dataset (ROCStories + TinyStories)
if not os.path.exists('data/combined/train.bin'):
    print("Building combined dataset...")
    !python data/combined/prepare.py
else:
    arr = np.fromfile('data/combined/train.bin', dtype=np.uint16)
    print(f"Combined dataset already exists: {len(arr)/1e6:.0f}M tokens")

In [ ]:
import os
os.chdir(LOCAL_DIR)

T4_CONFIG  = 'config/train_t4_arena.py'
T4_OUT_DIR = 'out-t4-arena'

ckpt = os.path.join(T4_OUT_DIR, 'ckpt.pt')
init_t4 = 'resume' if os.path.exists(ckpt) else 'scratch'

print(f"  ⚠  DO NOT upload this model for Task 3 — it exceeds 32M params!")
print(f"  This model is for the optional arena competition only.\n")
print(f"  Model    : 152M  (12L/12H/768D, all-modern LLaMA-style)")
print(f"  Dataset  : data/combined/  (ROCStories + TinyStories, ~110M tokens)")
print(f"  Config   : {T4_CONFIG}")
print(f"  Out dir  : {T4_OUT_DIR}")
print(f"  Init     : {init_t4}  ({'resuming from checkpoint' if init_t4 == 'resume' else 'training from scratch'})")
print(f"  Steps    : 20,000")
print(f"  ETA      : ~40–50 min on A100\n")

rc = run_streaming(
    ['python', '-u', 'train.py', T4_CONFIG, f'--init_from={init_t4}'],
    label="Task 4 Arena Model  ·  152M  —  ROCStories + TinyStories"
)

if rc == 0:
    print(f"\n✓  Arena model ready.  Checkpoint saved to: {T4_OUT_DIR}")
else:
    print(f"\n⚠  Process exited with code {rc}.")
    print("   Re-run this cell — it will resume from the last checkpoint automatically.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §4.3 — Task 4: Evaluation, Inference & Charts
# ═══════════════════════════════════════════════════════════════════════════
import json, math, os, torch
import matplotlib.pyplot as plt
import numpy as np
import tiktoken
os.chdir(LOCAL_DIR)

# ── 4.3.1 Load T4 checkpoint ──────────────────────────────────────────────
T4_DIR = 'out-t4-arena'
ckpt_path = os.path.join(T4_DIR, 'ckpt_best.pt')
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(T4_DIR, 'ckpt.pt')

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
from model import GPT, GPTConfig
cfg = GPTConfig(**ckpt['model_args'])
t4_model = GPT(cfg)
t4_model.load_state_dict(ckpt['model'])
t4_model.eval()

n_params = sum(p.numel() for p in t4_model.parameters())
print(f"T4 Arena model loaded")
print(f"  Architecture: {cfg.n_layer}L / {cfg.n_head}H / {cfg.n_embd}D")
print(f"  Parameters:   {n_params/1e6:.1f}M")
print(f"  Best val loss: {ckpt['best_val_loss']:.4f}  (PPL {math.exp(ckpt['best_val_loss']):.1f})")
print(f"  Trained for:  {ckpt['iter_num']} steps")

# ── 4.3.2 PPL on eval stories ─────────────────────────────────────────────
enc = tiktoken.get_encoding('gpt2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
t4_model = t4_model.to(device)

eval_prompts = [
    "Emily forgot her umbrella before leaving for work.",
    "Tom decided to cook dinner for his friends.",
    "Lily wanted to start jogging every morning.",
    "Mark borrowed his sister's bike for the afternoon.",
    "Anna planted tomato seeds in her backyard.",
]

def eval_ppl_on_text(model, texts, enc, device):
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            ids = enc.encode_ordinary(text)
            if len(ids) < 2: continue
            x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
            y = torch.tensor([ids[1:]],  dtype=torch.long, device=device)
            _, loss = model(x, y)
            total_loss   += loss.item() * (len(ids) - 1)
            total_tokens += (len(ids) - 1)
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    return avg_loss, math.exp(avg_loss)

eval_stories_path = 'data/rocstories/eval_stories.txt'
if os.path.exists(eval_stories_path):
    with open(eval_stories_path) as f:
        raw = f.read().strip()
    eval_texts = [s.strip() for s in raw.split('\n\n') if s.strip()]
else:
    eval_texts = eval_prompts
avg_loss_t4, ppl_t4 = eval_ppl_on_text(t4_model, eval_texts, enc, device)
print(f"\nT4 eval PPL on {len(eval_texts)} stories: {ppl_t4:.2f}")

# ── 4.3.3 Story generation ────────────────────────────────────────────────
T4_PARAMS = dict(max_new_tokens=200, temperature=0.75, top_k=50,
                 top_p=0.9, repetition_penalty=1.3)

print("\n── T4 Arena Generated Stories ──────────────────────────────────────")
t4_stories = []
for prompt in eval_prompts:
    ids = enc.encode_ordinary(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = t4_model.generate(idx, **T4_PARAMS)
    text = enc.decode(out[0].tolist())
    t4_stories.append({'prompt': prompt, 'generated': text})
    print(f"\nPrompt: {prompt}")
    print(f"Story:  {text[:300]}")

# ── 4.3.4 Learning curves ─────────────────────────────────────────────────
log_path = os.path.join(T4_DIR, 'train_log.jsonl')
if os.path.exists(log_path):
    t4_log = [json.loads(l) for l in open(log_path)]
    t4_val = [(e['step'], e['val_loss']) for e in t4_log if 'val_loss' in e]
    t4_train_steps = [e['step'] for e in t4_log if 'train_loss' in e and 'val_loss' not in e]
    t4_train_loss  = [e['train_loss'] for e in t4_log if 'train_loss' in e and 'val_loss' not in e]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Task 4 — Arena Model (152M params)', fontsize=13, fontweight='bold', y=1.02)

    ax = axes[0]
    if t4_train_steps:
        ax.plot(t4_train_steps, t4_train_loss, alpha=0.4, color='#B5D4F4', label='Train loss', linewidth=1)
    if t4_val:
        ax.plot([v[0] for v in t4_val], [v[1] for v in t4_val],
                color='#378ADD', label='Val loss', linewidth=2)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Training curves')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax2 = axes[1]
    if t4_val:
        val_ppls  = [math.exp(v[1]) for v in t4_val]
        val_steps = [v[0] for v in t4_val]
        ax2.plot(val_steps, val_ppls, color='#1D9E75', linewidth=2)
        best_step = val_steps[val_ppls.index(min(val_ppls))]
        best_ppl  = min(val_ppls)
        ax2.axvline(best_step, color='#E24B4A', linestyle='--', alpha=0.6, label=f'Best PPL={best_ppl:.1f}')
        ax2.axhline(25, color='#BA7517', linestyle=':', alpha=0.8, label='Pass line (25)')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Perplexity')
    ax2.set_title('Val PPL over training')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    ax3 = axes[2]
    model_labels, model_ppls, model_colors = [], [], []
    for out_dir, label, color in [
        ('out-t1-baseline', 'T1 Baseline\n(30M)', '#888780'),
        ('out-t3-best',     'T3 Best\n(30M)',     '#378ADD'),
        (T4_DIR,            'T4 Arena\n(152M)',   '#1D9E75'),
    ]:
        lp = os.path.join(out_dir, 'train_log.jsonl')
        if os.path.exists(lp):
            entries = [json.loads(l) for l in open(lp) if 'val_loss' in l]
            if entries:
                best_vl = min(entries, key=lambda e: e['val_loss'])['val_loss']
                model_labels.append(label)
                model_ppls.append(math.exp(best_vl))
                model_colors.append(color)
    if model_ppls:
        bars = ax3.bar(model_labels, model_ppls, color=model_colors, width=0.5)
        ax3.axhline(25, color='#E24B4A', linestyle='--', alpha=0.8, label='PPL=25 pass line')
        for bar, ppl in zip(bars, model_ppls):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{ppl:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax3.set_ylabel('Best Val PPL'); ax3.set_title('T1 vs T3 vs T4 comparison')
    ax3.legend(fontsize=9); ax3.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('t4_evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: t4_evaluation.png")

In [ ]:
# Run this cell in PARALLEL with any long training cell to prevent Colab idle disconnect
import time, threading
from IPython.display import display, Javascript

def keep_alive():
    while True:
        time.sleep(55)
        try:
            display(Javascript('console.log("keep-alive")'))
        except Exception:
            pass
        print(f"  [keep-alive] {time.strftime('%H:%M:%S')}", end="\r")

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("\u2713 Keep-alive thread started (pings every 55 s)")

✓ Keep-alive thread started (pings every 55 s)


---
## §5 · Final Summary — All Tasks

Aggregates results from Tasks 1–4. Run after all training is complete.

| Task | Model | Params | Expected Best PPL | Notes |
|------|-------|--------|-------------------|-------|
| T1 | Baseline (5K steps) | ~30M | ~28–31 | Pure ROCStories |
| T2-D | +QK-Norm ★ | ~30M | **~27–29** | Best individual modification |
| T3 | All Modern + 23M data (8K steps) | ~30M | **~20–24** | Should pass PPL 25 |
| T4 | 152M arena (20K steps) | 152M | **~12–18** | Arena only, no size limit |

*PPL < 25 threshold applies to T3 HuggingFace submission only.*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §5 — FINAL SUMMARY: Task 1 → Task 4
# ═══════════════════════════════════════════════════════════════════════════
import json, math, os, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
os.chdir(LOCAL_DIR)

# ── Collect all results ────────────────────────────────────────────────────
ALL_RUNS = [
    ('out-t1-baseline',   'T1: Baseline',         '#888780', False, False, False, False),
    ('out-t2-vanilla',    'T2-A: Vanilla',         '#888780', False, False, False, False),
    ('out-t2-rope',       'T2-B: +RoPE',           '#378ADD', True,  False, False, False),
    ('out-t2-ffn',        'T2-C: +RMSNorm+SwiGLU','#1D9E75', False, True,  True,  False),
    ('out-t2-qknorm',     'T2-D: +QK-Norm ★',     '#BA7517', False, False, False, True ),
    ('out-t2-all-modern', 'T2-E: All Modern',      '#7F77DD', True,  True,  True,  True ),
    ('out-t3-best',       'T3: Best Submission',   '#D85A30', True,  True,  True,  True ),
    ('out-t4-arena',      'T4: Arena (152M)',       '#0F6E56', True,  True,  True,  True ),
]

results = []
for out_dir, label, color, rope, rmsnorm, swiglu, qknorm in ALL_RUNS:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log):
        results.append({'label': label, 'color': color, 'best_ppl': None,
                        'best_step': None, 'rope': rope, 'rmsnorm': rmsnorm,
                        'swiglu': swiglu, 'qknorm': qknorm})
        continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries:
        continue
    best = min(entries, key=lambda e: e['val_loss'])
    ckpt_path = os.path.join(out_dir, 'ckpt_best.pt')
    if not os.path.exists(ckpt_path):
        ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    n_params = None
    if os.path.exists(ckpt_path):
        try:
            ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            from model import GPT, GPTConfig
            m = GPT(GPTConfig(**ck['model_args']))
            n_params = sum(p.numel() for p in m.parameters()) / 1e6
        except: pass
    results.append({
        'label': label, 'color': color,
        'best_ppl': math.exp(best['val_loss']),
        'best_step': best['step'],
        'best_val': best['val_loss'],
        'n_params': n_params,
        'rope': rope, 'rmsnorm': rmsnorm, 'swiglu': swiglu, 'qknorm': qknorm,
    })

# ── Print summary table ────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║              FINAL RESULTS SUMMARY — All Tasks                      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  {'Model':<30} {'Params':>7} {'Best PPL':>9} {'Best Step':>10} {'Flags':<15} ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
for r in results:
    if r['best_ppl'] is None:
        print(f"║  {r['label']:<30} {'—':>7} {'not run':>9} {'—':>10} {'—':<15} ║")
        continue
    params = f"{r['n_params']:.0f}M" if r.get('n_params') else "—"
    flags = '+'.join([f for f in [
        'R' if r['rope'] else '',
        'N' if r['rmsnorm'] else '',
        'S' if r['swiglu'] else '',
        'Q' if r['qknorm'] else '',
    ] if f])
    flags = flags if flags else 'vanilla'
    pass_marker = ' ✓' if r['best_ppl'] < 25 else '  '
    print(f"║  {r['label']:<30} {params:>7} {r['best_ppl']:>8.1f}{pass_marker} {r['best_step']:>10} {flags:<15} ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  Flags: R=RoPE  N=RMSNorm  S=SwiGLU  Q=QK-Norm  ✓=passes PPL<25   ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

# ── Figure 1: PPL bar chart (all tasks) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('COMP4680/8650 — NanoGPT ROCStories Results Summary',
             fontsize=13, fontweight='bold')

ax = axes[0]
valid = [r for r in results if r['best_ppl'] is not None]
labels = [r['label'].replace('T2-', '').replace(': ', '\n') for r in valid]
ppls   = [r['best_ppl'] for r in valid]
colors = [r['color'] for r in valid]
bars = ax.bar(range(len(valid)), ppls, color=colors, width=0.65, edgecolor='white', linewidth=0.5)
ax.axhline(25, color='#E24B4A', linestyle='--', linewidth=1.5, label='PPL=25 (pass line)')
for i, (bar, ppl) in enumerate(zip(bars, ppls)):
    ax.text(i, ppl + 0.3, f'{ppl:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(valid)))
ax.set_xticklabels(labels, fontsize=8, rotation=30, ha='right')
ax.set_ylabel('Best Validation PPL (lower = better)')
ax.set_title('Best val PPL per model (ckpt_best.pt)')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

ax2 = axes[1]
curve_configs = [
    ('out-t1-baseline',   'T1 Baseline',   '#888780'),
    ('out-t2-qknorm',     'T2-D +QK-Norm', '#BA7517'),
    ('out-t3-best',       'T3 Best',       '#D85A30'),
    ('out-t4-arena',      'T4 Arena',      '#1D9E75'),
]
for out_dir, label, color in curve_configs:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log): continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries: continue
    steps = [e['step'] for e in entries]
    ppls_c = [math.exp(e['val_loss']) for e in entries]
    ax2.plot(steps, ppls_c, label=label, color=color, linewidth=2)
    best_idx = ppls_c.index(min(ppls_c))
    ax2.scatter([steps[best_idx]], [ppls_c[best_idx]], color=color, s=50, zorder=5)
ax2.axhline(25, color='#E24B4A', linestyle='--', linewidth=1.5, alpha=0.8, label='PPL=25 pass line')
ax2.set_xlabel('Training step'); ax2.set_ylabel('Val PPL')
ax2.set_title('Learning curves — key models')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('final_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Architecture ablation detailed analysis ─────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))
fig2.suptitle('Task 2 — Architecture Ablation Analysis', fontsize=12, fontweight='bold')

abl_configs = [
    ('out-t2-vanilla',    'A. Vanilla',           '#888780'),
    ('out-t2-rope',       'B. +RoPE',              '#378ADD'),
    ('out-t2-ffn',        'C. +RMSNorm\n+SwiGLU', '#1D9E75'),
    ('out-t2-qknorm',     'D. +QK-Norm ★',        '#BA7517'),
    ('out-t2-all-modern', 'E. All Modern',         '#7F77DD'),
]

ax3 = axes2[0]
for out_dir, label, color in abl_configs:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log): continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries: continue
    steps = [e['step'] for e in entries]
    val_losses = [e['val_loss'] for e in entries]
    ax3.plot(steps, val_losses, label=label, color=color, linewidth=1.8)
ax3.set_xlabel('Step'); ax3.set_ylabel('Val loss')
ax3.set_title('Validation loss curves')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

ax4 = axes2[1]
abl_labels, abl_ppls, abl_colors = [], [], []
for out_dir, label, color in abl_configs:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log): continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries: continue
    best_vl = min(entries, key=lambda e: e['val_loss'])['val_loss']
    abl_labels.append(label); abl_ppls.append(math.exp(best_vl)); abl_colors.append(color)

if abl_ppls:
    bars = ax4.bar(range(len(abl_labels)), abl_ppls, color=abl_colors, width=0.6)
    baseline_ppl = abl_ppls[0] if abl_ppls else 30
    ax4.axhline(baseline_ppl, color='#888780', linestyle='--', alpha=0.6, label='Vanilla baseline')
    for i, (bar, ppl) in enumerate(zip(bars, abl_ppls)):
        ax4.text(i, ppl + 0.1, f'{ppl:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax4.set_xticks(range(len(abl_labels)))
ax4.set_xticklabels(abl_labels, fontsize=8, rotation=20, ha='right')
ax4.set_ylabel('Best Val PPL')
ax4.set_title('Best PPL by configuration\n(D = novel QK-Norm contribution)')
ax4.legend(fontsize=9); ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ablation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: final_summary.png, ablation_summary.png")

# ── Markdown-ready results for report ─────────────────────────────────────
print("\n── Copy-paste results for Task 2 report ─────────────────────────────")
print("| Config | Architecture | Best Val PPL | Best Step | vs Vanilla |")
print("|--------|-------------|-------------|-----------|------------|")
baseline_ppl_v = next((r['best_ppl'] for r in results if 'T2-A' in r['label']), None)
for r in results:
    if not r['label'].startswith('T2'): continue
    if r['best_ppl'] is None: continue
    delta = r['best_ppl'] - baseline_ppl_v if baseline_ppl_v else 0
    delta_str = f"{delta:+.1f}" if delta != 0 else "—"
    novel = " ★" if 'QK-Norm' in r['label'] else ""
    print(f"| {r['label'].split(': ')[1]+novel:<25} | "
          f"RoPE={'✓' if r['rope'] else '✗'} RMSNorm={'✓' if r['rmsnorm'] else '✗'} "
          f"SwiGLU={'✓' if r['swiglu'] else '✗'} QKNorm={'✓' if r['qknorm'] else '✗'} | "
          f"**{r['best_ppl']:.1f}** | {r['best_step']} | {delta_str} |")